In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

CREATE SCHEMA IF NOT EXISTS silver;

CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
import requests
import json

from pyspark.sql.functions import *
from datetime import datetime

In [0]:
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 9.894,
    "longitude": 76.718,

    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "wind_speed_10m",
        "cloud_cover"
    ],

    "timezone": "Asia/Kolkata",
    "forecast_days": 7
}

In [0]:
response = requests.get(
    url,
    params=params,
    timeout=30
)

response.raise_for_status()

weather_data = response.json()

print("API extraction successful!")

print(json.dumps(weather_data, indent=2)[:2000])

In [0]:
# Convert JSON to string

raw_json = json.dumps(weather_data)

# Create Spark DataFrame

bronze_df = spark.createDataFrame(
    [
        (
            raw_json,
            "Open-Meteo",
            "Thodupuzha"
        )
    ],
    [
        "raw_json",
        "source",
        "location"
    ]
)

# Add ingestion timestamp

bronze_df = bronze_df.withColumn(
    "ingestion_time",
    current_timestamp()
)

# Write into Bronze Delta table

bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("bronze.weather_raw")

print("Bronze data loaded successfully!")

In [0]:
%sql
SELECT *
FROM bronze.weather_raw;